In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/cleaned_results_2000.csv")

In [3]:

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")

df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result
0,2000-01-04,Egypt,Togo,2,1,Friendly,Aswan,Egypt,False,2000,home_win
1,2000-01-07,Tunisia,Togo,7,0,Friendly,Tunis,Tunisia,False,2000,home_win
2,2000-01-08,Trinidad and Tobago,Canada,0,0,Friendly,Port of Spain,Trinidad and Tobago,False,2000,draw
3,2000-01-09,Burkina Faso,Gabon,1,1,Friendly,Ouagadougou,Burkina Faso,False,2000,draw
4,2000-01-09,Guatemala,Armenia,1,1,Friendly,Los Angeles,United States,True,2000,draw


In [4]:
def expected_score(rating_a, rating_b):

    return 1 / (
        1 + 10 ** ((rating_b - rating_a) / 400)
    )

In [5]:
def update_elo(
    rating_a,
    rating_b,
    score_a,
    k=20
):

    expected_a = expected_score(
        rating_a,
        rating_b
    )

    return rating_a + k * (
        score_a - expected_a
    )

In [6]:
elo_ratings = {}

In [7]:
all_teams = set(
    df["home_team"]
).union(
    set(df["away_team"])
)

for team in all_teams:
    elo_ratings[team] = 1500

In [8]:
home_elos = []
away_elos = []
elo_diffs = []

In [9]:
for _, match in df.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]

    home_rating = elo_ratings[home_team]
    away_rating = elo_ratings[away_team]

    # Save PRE-MATCH ratings

    home_elos.append(home_rating)
    away_elos.append(away_rating)
    elo_diffs.append(
        home_rating - away_rating
    )

    # Determine result

    if match["home_score"] > match["away_score"]:

        home_score = 1
        away_score = 0

    elif match["home_score"] < match["away_score"]:

        home_score = 0
        away_score = 1

    else:

        home_score = 0.5
        away_score = 0.5

    # Update ratings

    new_home = update_elo(
        home_rating,
        away_rating,
        home_score
    )

    new_away = update_elo(
        away_rating,
        home_rating,
        away_score
    )

    elo_ratings[home_team] = new_home
    elo_ratings[away_team] = new_away

In [12]:
sorted(
    elo_ratings.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]

[('Argentina', 1942.67956599356),
 ('Spain', 1938.0684274656019),
 ('France', 1905.6172305208318),
 ('Brazil', 1872.493172506529),
 ('England', 1846.9905268671562),
 ('Portugal', 1846.2928275608663),
 ('Morocco', 1843.0831845541577),
 ('Japan', 1842.184938222792),
 ('Colombia', 1834.6416337939913),
 ('Germany', 1834.2264249290565),
 ('Netherlands', 1831.3984337620839),
 ('Croatia', 1797.0256008879862),
 ('Italy', 1795.6715415983742),
 ('Iran', 1790.971194939414),
 ('Mexico', 1789.4208694802799),
 ('Belgium', 1788.254369177261),
 ('Ecuador', 1786.6302686186282),
 ('Senegal', 1780.803675624259),
 ('Uruguay', 1779.887300429137),
 ('South Korea', 1777.2837850774604)]

In [13]:
latest_date = df["date"].max()

print(latest_date)

2026-05-31 00:00:00


In [14]:
team_matches = {}

for team in set(df["home_team"]).union(set(df["away_team"])):

    matches = pd.concat([
        df[df["home_team"] == team],
        df[df["away_team"] == team]
    ])

    team_matches[team] = matches.sort_values("date")

In [15]:
def get_previous_matches(team, match_date, n=5):

    matches = team_matches[team]

    previous = matches[
        matches["date"] < match_date
    ]

    return previous.tail(n)

In [16]:
def calculate_form(team, match_date):

    previous = get_previous_matches(team, match_date)

    if len(previous) < 5:
        return None

    points = 0

    for _, row in previous.iterrows():

        if row["home_team"] == team:

            if row["home_score"] > row["away_score"]:
                points += 3

            elif row["home_score"] == row["away_score"]:
                points += 1

        else:

            if row["away_score"] > row["home_score"]:
                points += 3

            elif row["away_score"] == row["home_score"]:
                points += 1

    return points / 15

In [17]:
def avg_goal_difference(team, match_date):

    previous = get_previous_matches(team, match_date)

    if len(previous) < 5:
        return None

    diffs = []

    for _, row in previous.iterrows():

        if row["home_team"] == team:
            diff = row["home_score"] - row["away_score"]
        else:
            diff = row["away_score"] - row["home_score"]

        diffs.append(diff)

    return sum(diffs) / len(diffs)

In [18]:
def avg_goals_scored(team, match_date):

    previous = get_previous_matches(team, match_date)

    if len(previous) < 5:
        return None

    goals = []

    for _, row in previous.iterrows():

        if row["home_team"] == team:
            goals.append(row["home_score"])
        else:
            goals.append(row["away_score"])

    return sum(goals) / len(goals)

In [19]:
def avg_goals_conceded(team, match_date):

    previous = get_previous_matches(team, match_date)

    if len(previous) < 5:
        return None

    goals = []

    for _, row in previous.iterrows():

        if row["home_team"] == team:
            goals.append(row["away_score"])
        else:
            goals.append(row["home_score"])

    return sum(goals) / len(goals)

In [20]:
team_rows = []

all_teams = set(df["home_team"]).union(
    set(df["away_team"])
)

for team in all_teams:

    form = calculate_form(
        team,
        latest_date
    )

    goal_diff = avg_goal_difference(
        team,
        latest_date
    )

    avg_scored = avg_goals_scored(
        team,
        latest_date
    )

    avg_conceded = avg_goals_conceded(
        team,
        latest_date
    )

    elo = elo_ratings.get(
        team,
        1500
    )

    if (
        form is None or
        goal_diff is None or
        avg_scored is None or
        avg_conceded is None
    ):
        continue

    team_rows.append({

        "team": team,

        "form": form,

        "goal_diff": goal_diff,

        "avg_scored": avg_scored,

        "avg_conceded": avg_conceded,

        "elo": elo

    })

In [21]:
team_stats = pd.DataFrame(
    team_rows
)

team_stats.head()

,team,form,goal_diff,avg_scored,avg_conceded,elo
0,Venezuela,0.466667,0.2,1.0,0.8,1656.596419
1,Yorkshire,0.800000,2.0,4.0,2.0,1538.046620
2,Basque Country,0.733333,1.2,2.0,0.8,1621.069826
3,Isle of Man,0.800000,3.2,4.6,1.4,1619.784118
4,Guam,0.400000,-0.6,1.8,2.4,1244.529359


In [22]:
team_stats = team_stats.set_index(
    "team"
)

In [23]:
team_stats.loc["Argentina"]

form               1.000000
goal_diff          3.000000
avg_scored         3.200000
avg_conceded       0.200000
elo             1942.679566
Name: Argentina, dtype: float64

In [24]:
team_stats.sort_values(
    "elo",
    ascending=False
).head(20)

,form,goal_diff,avg_scored,avg_conceded,elo
team,,,,,
Argentina,1.000000,3.0,3.2,0.2,1942.679566
Spain,0.733333,2.2,2.6,0.4,1938.068427
France,0.866667,1.8,2.8,1.0,1905.617231
Brazil,0.466667,0.4,1.8,1.4,1872.493173
England,0.666667,1.6,2.0,0.4,1846.990527
Portugal,0.533333,1.6,2.6,1.0,1846.292828
Morocco,0.733333,1.8,2.2,0.4,1843.083185
Japan,1.000000,1.6,2.0,0.4,1842.184938
Colombia,0.466667,0.2,1.4,1.2,1834.641634


In [25]:
team_stats.to_csv(
    "../data/team_stats.csv"
)

In [26]:
team_stats.loc["France"]

form               0.866667
goal_diff          1.800000
avg_scored         2.800000
avg_conceded       1.000000
elo             1905.617231
Name: France, dtype: float64

In [27]:
import joblib

joblib.dump(
    elo_ratings,
    "../models/elo_ratings.pkl"
)

['../models/elo_ratings.pkl']